# [2주차] [5. ML] 과제
- **과제 목적**: Titanic 데이터 하나로 처음부터 ML의 과정을 끝까지 직접 밟아봅니다. 라이브러리 함수를 호출하는 데서 끝내지 않고, 학습의 핵심(loss, gradient descent)과 평가의 핵심(confusion matrix와 4대 지표)을 **손으로 직접 계산**해 개념을 체화합니다. 이어서 두 모델을 동일 조건에서 비교하고 성능 차이의 원인을 모델 구조 관점에서 해석합니다. 마지막 심화에서는 수업에서 그림으로만 본 overfitting을 실험으로 직접 재현합니다.

> ⚠️ **주의**: LLM를 써도 좋지만, **판단·검증·해석은 반드시 직접** 수행하고 사용 내역을 `## 3`에 기록하세요.


## 0. 환경 설정

In [8]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

MY_SEED = 811  # TODO: 랜덤한 숫자
np.random.seed(MY_SEED)

#  Titanic 데이터를 사용
titanic = sns.load_dataset("titanic")
titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


## 1. 필수 과제

### 1-1. ML 문제 정의 (수업 §2)

코드를 짜기 전에, 이 문제를 ML 문제로 **정식화**합니다. 아래 빈칸을 채우세요.

- 우리는 이 데이터셋에서 타이타닉 탑승자의 생존 여부를 맞춰야합니다. **target(label)**은 어떤 column인가? →survived
- 이 데이터에는 label이 (있다). 따라서 학습 방식은 `지도`학습이다.
- target이 (범주형) 이므로, 이 문제는 (회귀 / 분류) 문제다. → 정답: `분류`
- 학습 전에 데이터를 train/test로 나누는 이유를 **본인의 언어로** 한 문장: `train/test로 나누어서 학습과 평가를 수행해야하기 때문이다.

### 1-2. 전처리와 데이터 분할

아래 TODO를 채우고, **각 주석의 빈칸에 '왜 이 단계가 필요한지'를 직접 적으세요.


In [9]:
# 사용할 feature와 target 선택
features = ["pclass", "sex", "age", "fare"]
target = "survived"

df = titanic[features + [target]].copy()

# [빈칸] age의 결측치를 중앙값으로 채우는 이유 (평균이 아니라 중앙값을 쓰는 이유 포함): 평균의 경우 극단적인 값에 영향을 받을 수 있지만 중앙값은 영향을 덜 받기 때문이다.
df["age"] = df["age"].fillna(df["age"].median())

# [빈칸] sex를 0/1 숫자로 바꾸는 이유 (모델 입장에서 설명): 값이 male, female 두 가지이므로 문자로 된 값을 모델이 처리할 수 있도록 0/1의 숫자로 변환하면 된다.
df["sex"] = (df["sex"] == "female").astype(int)

df = df.dropna()

X = df[features]
y = df[target]

# TODO: train/test를 8:2로 분할하세요. random_state=MY_SEED 사용
# [빈칸] stratify=y 옵션을 주는 이유: 생존과 사망 비율을 일정하게 유지하기 위해서 주는 옵션이다.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=MY_SEED,
    stratify=y)

print(f"train: {X_train.shape}, test: {X_test.shape}")
print(f"train 생존율: {y_train.mean():.3f}, test 생존율: {y_test.mean():.3f}")

train: (712, 4), test: (179, 4)
train 생존율: 0.383, test 생존율: 0.385


### 1-3. Gradient Descent 손계산 (수업 §3)

수업에서 "학습 = loss를 낮추는 방향으로 파라미터를 조금씩 갱신"이라고 했습니다. 이걸 **가장 작은 예제로 직접 계산**합니다.

**설정**: 데이터 3개 $(x, y) = (1,2), (2,4), (3,6)$, 모델 $\hat{y} = wx$ (절편 없음), loss는 MSE

$$L(w) = \frac{1}{3}\sum_{i=1}^{3}(y_i - wx_i)^2, \qquad \frac{\partial L}{\partial w} = -\frac{2}{3}\sum_{i=1}^{3} x_i(y_i - wx_i)$$

**문제**: $w_0 = 0$, learning rate $\eta = 0.1$에서 시작하여 **gradient descent를 3회 반복해 직접 계산**하고, **각 단계에서 $w$가 왜 그 방향으로, 왜 그만큼 움직였는지** 설명하세요. (계산기는 써도 되지만 코드로 먼저 답을 구하면 안 됩니다 — 아래 검증 셀은 손계산이 끝난 뒤에 실행)

| 반복 | 현재 $w$ | 각 데이터의 오차 $(y_i - wx_i)$ | gradient $\frac{\partial L}{\partial w}$ | 갱신된 $w$ |
|---|---|---|---|---|
| 1 | 0 | (2,4,6) | - 18.667 | 1.867 |
| 2 | 1.867 | (0.133, 0.267, 0.400) | -1.244 | 1.991 |
| 3 | 1.991 | (0.009, 0.018, 0.027) | -0.083 | 1.999 |

**서술**: 매 반복마다 gradient의 크기가 어떻게 변했고, 그것이 $w$의 이동량과 어떤 관계인가? 이 과정이 언제, 왜 멈추게 될까? → `반복이 진행될수록 gradient의 크기가 줄어들어 w의 이동량이 감소합니다. 예측 오차가 0에 가까워지면 gradient가 0이 되어 w가 최적값에 수렴하므로 갱신이 멈추게 됩니다.`

In [10]:
# ===== 손계산 검증용 셀 (표를 다 채운 뒤에 실행하세요) =====
# 주의: 위에서 만든 target y(Series)를 덮어쓰지 않도록 여기서는 gx, gy 라는 별도 변수를 쓴다.
gx = np.array([1, 2, 3])
gy = np.array([2, 4, 6])

w = 0.0
lr = 0.1

for step in range(1, 4):
    # TODO: 위 수식대로 gradient를 코드로 옮기세요
    grad = -2/len(gx) * np.sum(gx * (gy - w *gx))   # 힌트: -2/len(gx) * np.sum(...)

    # [빈칸] gradient의 '반대 방향'으로 이동하는 이유: gradient는 loss가 가장 빠르게 증가하는 방향을 나타내므로 loss를 줄이기 위해 반대 방향으로 이동한다.
    w = w - lr * grad

    loss = np.mean((gy - w * gx) ** 2)
    print(f"step {step}: grad = {grad:+.4f}, w = {w:.4f}, loss = {loss:.4f}")

# 손계산 결과와 출력이 일치하는지 확인하고, 다르면 어디서 틀렸는지 찾아 적으세요: 일치한다

step 1: grad = -18.6667, w = 1.8667, loss = 0.0830
step 2: grad = -1.2444, w = 1.9911, loss = 0.0004
step 3: grad = -0.0830, w = 1.9994, loss = 0.0000


### 1-4. 두 모델 학습과 비교

수업에서 "모든 알고리즘은 (f의 형태 / loss / 찾는 방법)에 대한 서로 다른 답"이라고 했습니다. 형태가 전혀 다른 두 모델을 **같은 데이터, 같은 조건**에서 학습시켜 비교합니다.

- **모델 A**: Logistic Regression (선형 + sigmoid)
- **모델 B**: Decision Tree (재귀 분할)


In [11]:
# 과정 확인 장치: train과 test 성능을 모두 기록합니다
results = {}

# TODO: 모델 A — LogisticRegression을 학습시키세요 (max_iter=1000)
model_a = LogisticRegression(max_iter=1000)
model_a.fit(X_train, y_train)

# TODO: 모델 B — DecisionTreeClassifier를 학습시키세요 (random_state=MY_SEED)
model_b = DecisionTreeClassifier(random_state=MY_SEED)
model_b.fit(X_train, y_train)

for name, model in [("Logistic Regression", model_a), ("Decision Tree", model_b)]:
    # [빈칸] train 정확도와 test 정확도를 '둘 다' 기록하는 이유 (수업 §3의 개념과 연결해서 : 모델이 학습 데이터에서는 잘 맞지만 새로운 테스트 데이터에서는 성능이 떨어진다면 overfitting을 의심할 수 있다. 따라서 과적합을 확인하기 위해서 train, test 정확도를 둘 다 기록해야한다.
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    results[name] = (train_acc, test_acc)
    print(f"{name:20s} | train acc: {train_acc:.4f} | test acc: {test_acc:.4f}")

Logistic Regression  | train acc: 0.7739 | test acc: 0.8547
Decision Tree        | train acc: 0.9846 | test acc: 0.7877


### 1-5. 평가지표 직접 유도

수업에서 "Confusion Matrix 하나에서 4개 지표가 전부 유도된다"고 했습니다. **sklearn의 지표 함수를 쓰기 전에**, TP/TN/FP/FN으로부터 직접 계산해 봅니다. (모델 A 기준)


In [12]:
y_pred = model_a.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

# sklearn의 confusion_matrix 배치: [[TN, FP], [FN, TP]]
TN, FP = cm[0]
FN, TP = cm[1]
print(f"TP={TP}, TN={TN}, FP={FP}, FN={FN}")

# TODO: 아래 4개 지표를 TP/TN/FP/FN '만으로' 직접 계산하세요 (sklearn 함수 사용 금지)
my_accuracy  = (TP + TN) / (TP + TN + FP + FN)
my_precision = TP / (TP + FP)  # [빈칸] precision의 분모에 들어가는 것과 그 의미: TP + FP, Positive라고 예측한 전체 개수
my_recall    = TP / (TP + FN) # [빈칸] recall의 분모에 들어가는 것과 그 의미: TP + FN, 실제 Positive인 전체 개수
my_f1        = 2 * my_precision * my_recall /( my_precision + my_recall )  # 힌트: precision과 recall의 조화평균

# 검증: sklearn과 비교 (통과하지 못하면 수식을 다시 확인)
assert np.isclose(my_accuracy,  accuracy_score(y_test, y_pred))
assert np.isclose(my_precision, precision_score(y_test, y_pred))
assert np.isclose(my_recall,    recall_score(y_test, y_pred))
assert np.isclose(my_f1,        f1_score(y_test, y_pred))
print("✅ 4개 지표 모두 sklearn과 일치")

Confusion Matrix:
 [[95 15]
 [11 58]]
TP=58, TN=95, FP=15, FN=11
✅ 4개 지표 모두 sklearn과 일치


### 1-6. 결과 해석 (필수 서술)

아래 4개 질문에 **본인의 실험 결과 수치를 근거로** 답하세요. (수치 없이 일반론만 쓰면 감점)

1. **왜 이러한 결과가 나왔는가?** — 두 모델의 test 정확도는 각각 얼마였고, 이 데이터에서 그 정도 성능이 나온 이유를 feature 관점에서 추측하면? → `Logistic Regression의 test acc는 0.8101 였고, Decision Tree의test acc는 0.7877였다.feature로 "pclass", "sex", "age", "fare"를 사용했는데, 이는 승객의 객실 등급,성별,나이,요금 등 타이타닉 데이터셋에서 생존율에 결정적인 영향을 미치는 주요 변수들이 포함되었기 때문에 두 모델 모두 80% 내외의 classification 성능을 기록할 수 있었다.`
2. **두 모델의 성능 차이는 어디에서 발생했는가?** — train acc와 test acc의 '차이'가 두 모델에서 어떻게 달랐는가? 이를 모델 구조(선형 경계 vs 재귀 분할)의 관점에서 해석하면? → `Train acc와 test acc의 차이를 비교해보면, Logistic Regression은 Train(0.7823)보다 Test(0.8101)가 약간 높게 유지된 반면, Decision Tree는 Train(0.9775)이 매우 높았지만 Test(0.7877)는 크게 떨어져 약 19%p에 달하는 심한 격차를 보였다. 선형 경계 기반의 Logistic Regression은 데이터 경계를 단순화하여 일반화 성능을 유지한 반면, 재귀 분할 방식의 Decision Tree는 Train 데이터의 노이즈까지 지나치게 적응하여 Overfitting이 크게 발생했기 때문이다.`
3. **실험 조건을 변경하면 결과가 어떻게 달라지는가?** — `MY_SEED`를 다른 값으로 바꿔 1-2 ~ 1-4를 다시 실행해 보고, 어떤 수치가 얼마나 흔들렸는지 기록. 이 흔들림을 줄이는 방법으로 수업에서 배운 것은? → `MY_SEED를 변경함에 따라 데이터의 train/test 분할 양상이 바뀌면서 성능 수치에 변화가 생겼다.
Logistic Regression test acc가 기존 0.8101에서 0.8547로 약 4.46%p 대폭 상승했다. Decision Tree test acc는 0.7877로 동일하게 유지되었으나 train acc가 0.9775에서 0.9846으로 소폭 상승했다. 이처럼 데이터셋 크기가 작을 때는 데이터 분할에 따라 특정 모델의 test 성능이 민감하게 흔들리는 현상이 발생한다. 이러한 무작위 분할에 따른 성능 평가의 흔들림(변동성)을 줄이고 모델의 안정적인 일반화 성능을 구하기 위해 수업에서 배운 교차 검증(Cross Validation, K-Fold)을 활용할 수 있다.`
4. **예상과 다른 결과가 나왔다면 그 원인은 무엇인가?** — 과제를 시작하기 전 예상과 달랐던 지점 하나와, 그 원인에 대한 본인의 가설: → `복잡한 규칙을 만드는 Decision Tree가 더 높은 성능을 낼 것이라는 예상과 달리, 단순한 선형 모델인 Logistic Regression이 더 높은 성능을 기록했다.
그 원인은 Decision Tree의 극심한 과적합에 있다. Decision Tree는 train acc가 0.9846에 달하지만 test acc는 0.7877에 그쳐 심한 과적합을 나타냈다. 반면 feature 수가 4개로 적고 데이터 수가 제한적인 상황에서는, 복잡한 트리 분할보다 Logistic Regression의 단순한 선형 경계가 노이즈 학습을 방지하여 새로운 test 데이터에 훨씬 더 강력한 일반화 성능을 발휘했기 때문이다.`


## 2. 심화 과제 (선택)

### 2-1. Overfitting 직접 재현하기

 overfitting/underfitting curve를 **본인 데이터로 직접 그립니다.** Decision Tree의 `max_depth`를 1부터 15까지 바꿔가며 train/test 정확도를 기록하세요.


In [13]:
depths = range(1, 16)
train_scores, test_scores = [], []

for d in depths:
    # TODO: max_depth=d인 DecisionTreeClassifier를 학습시키고
    #       train/test 정확도를 각 리스트에 추가하세요
    ______

plt.figure(figsize=(8, 5))
plt.plot(depths, train_scores, marker="o", label="train")
plt.plot(depths, test_scores, marker="s", label="test")
# Colab 기본 환경엔 한글 폰트가 없어 제목/라벨은 영문으로 둔다(한글이면 □로 깨짐)
plt.xlabel("max_depth (model complexity)")
plt.ylabel("accuracy")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

# 해석 (필수):
# - 그래프에서 underfitting 구간과 overfitting 구간은 각각 어디인가? → ______
# - 본인 데이터 기준 최적의 max_depth와 그렇게 판단한 근거는? → ______
# - 이 그래프가 수업의 "train loss만 낮추는 건 쉽다"는 문장과 어떻게 연결되는가? → ______

NameError: name '______' is not defined

### 2-2. 비지도학습 미리보기 — label을 가리면 무엇이 보일까?

지금까지는 `survived`라는 label을 알고 학습했습니다(지도학습). 이번엔 **label을 일부러 가리고**, 데이터의 구조만으로 무엇이 보이는지 확인합니다.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# [빈칸] PCA/K-means 전에 스케일링이 필요한 이유 (fare와 age의 단위를 생각해 보세요): ______
X_scaled = StandardScaler().fit_transform(X)

# TODO: PCA로 2차원으로 축소하세요
X_2d = ______

# TODO: K-means로 2개 군집을 찾으세요 (random_state=MY_SEED, n_init=10)
clusters = ______

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(X_2d[:, 0], X_2d[:, 1], c=clusters, cmap="coolwarm", alpha=0.6, s=15)
axes[0].set_title("K-means clusters (no label)")
# ⚠️ c=y 로 넘기면 pandas Series를 단일 RGBA로 오해해 ValueError가 난다. 반드시 c=y.values
axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=y.values, cmap="coolwarm", alpha=0.6, s=15)
axes[1].set_title("Actual survived (label)")
plt.show()

# 해석 (필수):
# - K-means가 찾은 군집은 실제 생존 여부와 얼마나 겹치는가? → ______
# - 겹친다면/겹치지 않는다면, 그 이유는 무엇이라고 생각하는가?
#   (힌트: K-means는 무엇을 기준으로 데이터를 나누는가?) → ______

## 3. 생성형AI 활용 방법

문제를 풀면서 GPT, Claude 등 생성형 AI를 **어디에, 어떻게** 썼는지 기록하는 곳입니다. (사용하지 않았다면 "사용하지 않음"이라고 적으세요)

- **활용 방법**: 어떤 문제에서, 어떤 목적으로(개념 질문 / 에러 해결 / 코드 초안 등) 사용했는지 → `개념 질문`
- **AI 답변 중 그대로 쓰지 않고 직접 수정·검증한 부분**: → `gradient의 반대 방향으로 이동하는 이유`
- **대화 내역 붙여넣기**: https://share.gemini.google/Iki1LeRswZAy
```
2번 빈칸  여기엔 원핫인코딩이나 더미 변환 사용한 건 아니야?

아닙니다. 해당 코드는 원-핫 인코딩(One-Hot Encoding)이나 더미 변환(Dummy Variable Creation)이 아니라, 이진 인코딩(Binary Encoding) / 레이블 인코딩(Label Encoding) 방식을 사용한 것입니다.

차이점 분석

현재 사용된 코드: (df["sex"] == "female").astype(int)

sex 컬럼 하나 안에서 female은 1, male은 0이라는 단일 수치형 값으로 1:1 변환합니다.

새로운 열을 생성하지 않고 기존 열의 데이터 타입만 숫자로 바꿉니다.

원-핫 인코딩 / 더미 변환을 했을 경우 (pd.get_dummies)

sex_female, sex_male과 같이 범주(카테고리)의 개수만큼 새로운 열(Column)을 여러 개 생성합니다.

[빈칸] sex를 0/1 숫자로 바꾸는 이유 (모델 입장에서 설명)

직접적인 이유: 대부분의 머신러닝 모델(수학적 알고리즘)은 문자열("male", "female") 데이터를 직접 연산할 수 없기 때문에 수치형 데이터로 변환해야 합니다.

0/1로 변환한 이유: 범주가 딱 2개(이진 카테고리)뿐이므로, 열을 새로 늘리는 원-핫 인코딩 대신 하나의 열에 0과 1을 매핑하여 차원(열)을 늘리지 않고 효율적으로 수치화할 수 있습니다.

gradient의 반대방향으로 이동하는 이유

경사하강법(Gradient Descent)에서 경사(Gradient, ∇f)의 반대 방향(−∇f)으로 이동하는 이유는 ∇f가 함수 값이 '가장 가파르게 증가하는 방향'을 가리키기 때문입니다. 따라서 함수 값을 최소화하려면 그와 반대 방향인 '가파르게 감소하는 방향'으로 가야 합니다.

이를 수학적·직관적으로 나누어 보면 다음과 같습니다.

1. 직관적인 이유: 산 내려오기

Gradient (∇f): 현재 위치에서 가장 가파른 오르막길 방향

Negative Gradient (−∇f): 현재 위치에서 가장 가파른 내리막길 방향

손실 함수(Loss function)의 값을 최소화하는 것이 목표이므로, 가장 빠르게 아래로 내려가기 위해 negative gradient 방향을 선택합니다.

2. 수학적인 이유: 방향유도(Directional Derivative)와 내적

어떤 위치 $\mathbf{x}$에서 단위 벡터 v 방향으로 이동할 때 함수 $f(\mathbf{x})$의 변화율(방향유도)은 두 벡터의 내적(Inner Product)으로 표현됩니다.

∇
v
​
 f(x)=∇f(x)⋅v=∥∇f(x)∥∥v∥cos(θ)
θ: Gradient 벡터 $\nabla f(\mathbf{x})$와 이동 방향 v 사이의 각도

이 변화율을 최소(가장 큰 음수, 즉 가장 가파른 감소)로 만들려면 cos(θ) 값이 최소가 되어야 합니다.
$\cos(\theta)$의 최솟값은 \theta = 180^\circ (\pi)$일 때 $-1입니다.

즉, 이동 방향 $\mathbf{v}$가 Gradient 방향 \nabla f(\mathbf{x})$와 정반대 방향($-\nabla f)일 때 함수 값이 가장 빠르게 감소합니다.

요약

Gradient 방향이 "어디로 가야 가장 빠르게 올라가는가"를 알려주는 이정표이기 때문에, 반대 방향인 −∇f는 "어디로 가야 가장 빠르게 내려가는가"를 알려주므로 그 방향으로 이동합니다.
```


## 4. 회고

- **가장 어려웠던 부분**과 그것을 어떻게 해결했는지 (또는 아직 해결하지 못했는지, 없다면 없음이라고 적어도 됨): → `Gradient Descent의 weight 갱신 과정을 직접 계산하는 부분이 가장 어려웠습니다. 이를 해결하기 위해 관련 개념을 다시 복습하였고, 매 step 마다 어떻게 변하는지 종이에 차근차근 계산하여 해결하였습니다. `
- 이번 과제 내용 중 기술블로그 '과제 복습' 섹션에 정리할 핵심 1가지: → `Decision Tree의 과적합(Overfitting)과 Logistic Regression의 일반화(Generalization) 성능 비교`
